In [1]:
from scipy.stats import norm
from scipy.stats import chi2

from statsmodels.formula.api import glm
from statsmodels.genmod.families import Poisson
from statsmodels.genmod.families import Binomial
from statsmodels.stats.contingency_tables import Table2x2
from statsmodels.graphics.mosaicplot import mosaic

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os
from itertools import product
pd.set_option("display.width", 2000)

# import custom functions
from ST5213_helpers import *

In [3]:
# G = np.array(['M', 'M', 'M', 'M', 'F', 'F', 'F', 'F'])
# I = np.array(['S', 'S', 'O', 'O', 'S', 'S', 'O', 'O'])
# H = np.array(['S', 'O', 'S', 'O', 'S', 'O', 'S', 'O'])
# Count = np.array([76, 160, 6, 25, 114, 181, 11, 48])

# df = pd.DataFrame({'G': G, 'I': I, 'H': H, 'Count': Count})

df = pd.DataFrame(list(product(['M', 'F'], ['S', 'O'], ['S', 'O'])), columns=['G', 'I', 'H'])
df['Count'] = [76, 160, 6, 25, 114, 181, 11, 48]
df[['G', 'I', 'H']] = df[['G', 'I', 'H']].astype('category')

fm1 = glm('Count ~ G*H + G*I', data=df, family=Poisson()).fit()
fm2 = glm('Count ~ G*H + H*I', data=df, family=Poisson()).fit()
fm3 = glm('Count ~ G*I + H*I', data=df, family=Poisson()).fit()
fm4 = glm('Count ~ G*H + G*I + H*I', data=df, family=Poisson()).fit()

In [4]:
print(drop1_glm(fm4))

       Df   Deviance        AIC        LRT  Pr(>Chi)
<none>      0.300721  59.682728                     
G:H     1   2.383139  59.765145   2.082418  0.149004
G:I     1   4.126712  61.508718   3.825991  0.050464
H:I     1  11.665735  69.047741  11.365014  0.000748


In [6]:
print("fm1: ", fm1.aic)
print("fm2: ", fm2.aic)
print("fm3: ", fm3.aic)
print("fm4: ", fm4.aic)

fm1:  69.04774136775328
fm2:  61.508718260146445
fm3:  59.76514525150965
fm4:  59.682727564521414


In [7]:
print("fm1: ", chi2.sf(fm1.deviance, fm1.df_resid))
print("fm2: ", chi2.sf(fm2.deviance, fm2.df_resid))
print("fm3: ", chi2.sf(fm3.deviance, fm3.df_resid))
print("fm4: ", chi2.sf(fm4.deviance, fm4.df_resid))

fm1:  0.002929663961957288
fm2:  0.12702695163005986
fm3:  0.30374416537301624
fm4:  0.583430583877323


In [8]:
print(fm4.summary2())

              Results: Generalized linear model
Model:               GLM              Method:         IRLS   
Model Family:        Poisson          AIC:            59.6827
Link Function:       Log              BIC:            60.2388
Dependent Variable:  Count            Log-Likelihood: -22.841
Date:                2026-10-20 12:12 LL-Null:        -245.60
No. Observations:    8                Deviance:       0.30072
Df Model:            6                Pearson chi2:   0.307  
Df Residuals:        1                Scale:          1.0000 
-------------------------------------------------------------
               Coef.  Std.Err.    z    P>|z|   [0.025  0.975]
-------------------------------------------------------------
Intercept      3.8521   0.1415 27.2194 0.0000  3.5748  4.1295
G[T.M]        -0.5976   0.2242 -2.6658 0.0077 -1.0370 -0.1582
H[T.S]        -1.3750   0.2750 -5.0008 0.0000 -1.9139 -0.8361
I[T.S]         1.3514   0.1575  8.5777 0.0000  1.0426  1.6601
G[T.M]:H[T.S] -0.2516 

In [9]:
beta = np.array(fm4.params)
SE = np.array(fm4.bse)

GH_CI = (beta[4] - norm.ppf(0.975)*SE[4], beta[4] + norm.ppf(0.975)*SE[4])
print(np.exp(GH_CI))

GI_CI = (beta[5] - norm.ppf(0.975)*SE[5], beta[5] + norm.ppf(0.975)*SE[5])
print(np.exp(GI_CI))

[0.55184997 1.09555131]
[0.99204747 2.54774644]


In [10]:
np.exp(-0.2516), np.exp(0.4636)

(np.float64(0.7775556981520442), np.float64(1.589786928468738))

In [12]:
fm4 = update_glm(fm4, drop='G:H')
print(drop1_glm(fm4))

       Df   Deviance        AIC        LRT  Pr(>Chi)
<none>      2.383139  59.765145                     
G:I     1   5.581035  60.963041   3.197896  0.073733
H:I     1  13.120058  68.502064  10.736919   0.00105


In [13]:
fm4 = update_glm(fm4, drop='G:I')
print(drop1_glm(fm4))

       Df   Deviance        AIC        LRT  Pr(>Chi)
<none>      5.581035  60.963041                     
G       1  17.809627  71.191634  12.228593  0.000471
H:I     1  16.317954  69.699960  10.736919   0.00105


In [14]:
chi2.sf(fm4.deviance, fm4.df_resid)

np.float64(0.13387138524958608)